# Python Fundamentals — Beginner Notes
### Week 1: Core Data Structures & Functions

**Goal of this notebook:** Don't just learn Python *syntax* — understand *why* Python behaves the way it does,
by comparing it to how a lower-level language like **C** handles the same problem. Python hides a lot of
"plumbing" from you. Seeing the plumbing (even briefly) makes the Pythonic way click.

Each section has:
1. **Concept explanation** (Python vs C mindset)
2. **Worked examples** (run every cell, tweak values, break things on purpose)
3. A short **checkpoint** question

At the very end: a **Week Assessment** — oral questions (answer in your head or out loud) + coding exercises
(with solutions hidden below so you can self-check).

## 0. The Big Picture: Why Python "just works" (and C doesn't, automatically)

In **C**, YOU are the memory manager:

```c
// C: you decide how much memory, you allocate it, you free it
int *arr = malloc(5 * sizeof(int));   // ask the OS for exactly 20 bytes
arr[0] = 10;
// ... use it ...
free(arr);                             // YOU must give it back, or it's a memory leak
```

If you forget `free()`, that memory is leaked forever (until the program ends).
If you access `arr[10]` (out of bounds), C will **not stop you** — it's "undefined behavior":
maybe garbage, maybe a crash, maybe silent corruption of other data. C trusts you completely.

In **Python**, every value (even an integer!) is an **object** living on the heap, and Python's
runtime (CPython) automatically:
- Allocates memory when you create an object
- Tracks how many things "point to" (reference) that object — called **reference counting**
- Frees the memory automatically the moment nothing references it anymore (plus a garbage
  collector for tricky circular references)

```python
x = [1, 2, 3]   # Python allocates a list object, x refers to it
x = None        # nothing refers to the list anymore -> Python frees it automatically
```

You never call `malloc`/`free` in Python. This is *why* Python feels "batteries included" —
lists can grow, strings can be any length, dictionaries just work — because Python is doing the
manual memory bookkeeping *for* you, behind the scenes, every single time.

**Keep this contrast in mind for every topic below**: whenever Python "just handles it", ask
yourself "what would I have to do manually in C to get this same behavior?"

## 1. Strings

**C mindset:** A string is just an array of `char`, ending with a special `'\0'` (null) byte.
```c
char name[6] = "Hello";   // you must size the array yourself (5 chars + 1 for '\0')
// name[5] = 'X';         // no automatic bounds checking!
```
Strings in C are **mutable** (you can change individual chars) but you must manage the buffer size
yourself. Concatenating strings means manually allocating a bigger buffer and copying bytes over
(`strcat`, `strcpy` — classic sources of bugs like buffer overflows).

**Python mindset:** A string is an **immutable** object. Python manages the exact memory needed,
handles Unicode automatically, and every "modification" actually creates a **brand-new string
object** rather than editing the old one in place.

In [ ]:
s = "Hello"
print(s, type(s), len(s))

# Strings are IMMUTABLE -> this would error:
try:
    s[0] = "J"
except TypeError as e:
    print("Error:", e)

# "Modifying" a string really creates a new object
s2 = s.replace("H", "J")
print("original:", s, "| new:", s2)
print("same object?", s is s2)   # False -> two separate objects in memory

In [ ]:
# Common string operations
s = "Python Programming"

print(s.upper())
print(s.lower())
print(s.split())            # splits on whitespace by default -> list
print(s.replace("Python", "C"))
print(s.find("Prog"))       # index where substring starts, -1 if not found
print(s.strip())            # removes leading/trailing whitespace
print("  padded  ".strip())
print(" - ".join(["a", "b", "c"]))   # opposite of split
print(f"Length is {len(s)}, first word is '{s.split()[0]}'")   # f-strings

**Checkpoint:** Why does `s[0] = 'J'` fail in Python but `name[0] = 'J'` would work in C?

## 2. Lists

**C mindset:** Arrays are **fixed size** and set at creation. Want a bigger array? You must
`malloc` a new, bigger block and copy every element over yourself.
```c
int arr[5] = {1,2,3,4,5};
// arr[5] = 6;   // NOT ALLOWED - out of bounds, undefined behavior, possibly a crash later
```

**Python mindset:** A `list` is a **dynamic array** — Python automatically resizes the underlying
memory block for you (over-allocating a bit so repeated `append()` calls stay fast). You never
think about capacity.

In [ ]:
nums = [1, 2, 3]
nums.append(4)          # Python handles resizing internally
nums.insert(0, 0)       # insert at index 0, shifts everything else
print(nums)

nums.remove(3)          # removes the VALUE 3 (first occurrence)
popped = nums.pop()     # removes & returns the LAST element
print(nums, "popped:", popped)

nums.extend([10, 20])   # merge another list in
print(nums)

print(len(nums), 5 in nums, nums.count(0))
nums.sort()
print("sorted:", nums)
nums.reverse()
print("reversed:", nums)

**Checkpoint:** In C you'd need to know the array's max size in advance. What does Python do internally so you never have to?

## 3. Tuples

**C mindset:** Closest equivalent is a fixed `struct` or a `const` array — a fixed-size,
fixed-content group of values that (ideally) shouldn't change.

**Python mindset:** A `tuple` is like a list, but **immutable** — once created, it cannot be
resized or have its elements reassigned. Because it's immutable, Python can store it more
compactly and it's safe to use as a dictionary key (lists cannot be, since they're mutable).

In [ ]:
point = (3, 4)
print(point, type(point))

try:
    point[0] = 99
except TypeError as e:
    print("Error:", e)

# Tuple unpacking - very common Python idiom
x, y = point
print(f"x={x}, y={y}")

# tuples as dict keys (lists CAN'T do this - see Dictionaries section)
locations = {(0, 0): "origin", (1, 1): "diagonal"}
print(locations[(0, 0)])

# single-element tuple needs a trailing comma!
one_elem = (5,)
not_a_tuple = (5)
print(type(one_elem), type(not_a_tuple))

**Checkpoint:** Why can a tuple be used as a dictionary key, but a list cannot?

## 4. Sets

**C mindset:** There's no built-in set type. To get "unique items with fast lookup" you'd have to
hand-write a **hash table** yourself: pick a hash function, handle collisions, resize the table
when it fills up. That's a whole data-structures assignment on its own!

**Python mindset:** `set` is a built-in hash table. Membership testing (`x in my_set`) is on
average **O(1)** — instant — regardless of size, versus **O(n)** (checking one by one) for a list.

In [ ]:
fruits = {"apple", "banana", "apple", "cherry"}   # duplicates auto-removed
print(fruits)                                      # order not guaranteed

fruits.add("mango")
fruits.discard("banana")   # safe removal, no error if missing
print(fruits)

a = {1, 2, 3, 4}
b = {3, 4, 5, 6}
print("union:", a | b)
print("intersection:", a & b)
print("difference (a-b):", a - b)
print("symmetric diff:", a ^ b)

# Why sets are fast: membership check
big_list = list(range(100000))
big_set = set(big_list)
print(99999 in big_set)    # fast, hash lookup
print(99999 in big_list)   # slow-ish, checks one by one (still fine here, just illustrating)

**Checkpoint:** Why is `x in my_set` typically much faster than `x in my_list` for large collections?

## 5. Dictionaries

**C mindset:** Same story as sets — no built-in "key → value" structure. You'd hand-build a
hash table where you hash the key to find a bucket, then store the value there, handling
collisions manually with linked lists or open addressing.

**Python mindset:** `dict` is a built-in, highly optimized hash map. Keys must be **hashable**
(immutable-ish: strings, numbers, tuples of immutables) — which is exactly why lists can't be
keys but tuples can.

In [ ]:
student = {"name": "Deepak", "age": 24, "skills": ["Python", "FastAPI"]}

print(student["name"])
print(student.get("gpa", "N/A"))     # .get avoids KeyError, gives a default

student["age"] = 25                  # update
student["city"] = "Hyderabad"        # add new key
del student["city"]                  # remove

for key, value in student.items():
    print(f"{key} -> {value}")

print(list(student.keys()))
print(list(student.values()))

try:
    print(student["gpa"])            # KeyError! key doesn't exist
except KeyError as e:
    print("Error: missing key", e)

**Checkpoint:** What error do you get accessing a missing dict key with `[]`? How does `.get()` avoid it?

## 6. Mutability

This is the concept that trips up the most beginners, so let's be very explicit.

**C mindset:** Everything is mutable by default unless you mark it `const`. Mutability is about
*memory permissions*, enforced by the compiler at the byte level.

**Python mindset:** Mutability is a property of the **type itself**:

| Mutable (can change in place) | Immutable (cannot change in place) |
|---|---|
| `list`, `dict`, `set` | `str`, `tuple`, `int`, `float`, `bool`, `frozenset` |

Variables in Python are **names bound to objects**, not boxes holding values. This matters a lot
when you pass mutable objects around — two names can point to the *same* object.

In [ ]:
# Immutable example: int
a = 5
b = a
b = b + 1          # this creates a NEW int object for b; a is untouched
print(a, b)

# Mutable example: list
list_a = [1, 2, 3]
list_b = list_a          # list_b points to the SAME list object as list_a!
list_b.append(4)
print("list_a:", list_a) # changed too! because they're the same object
print("list_b:", list_b)
print("same object?", list_a is list_b)

# To actually COPY a list:
list_c = list_a.copy()   # or list(list_a) or list_a[:]
list_c.append(99)
print("list_a:", list_a)
print("list_c:", list_c)

In [ ]:
# The classic beginner trap: mutable default arguments
def add_item(item, basket=[]):   # DANGER: default list is created ONCE, shared across calls!
    basket.append(item)
    return basket

print(add_item("apple"))
print(add_item("banana"))   # you'd expect just ['banana'], but you get both!

# Fix: use None and create a fresh list each call
def add_item_fixed(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item_fixed("apple"))
print(add_item_fixed("banana"))

**Checkpoint:** If `list_b = list_a` and you `.append()` to `list_b`, why does `list_a` change too?

## 7. Indexing / Slicing

**C mindset:** `arr[i]` is literally `*(arr + i)` — pointer arithmetic, no bounds checking.
Reading `arr[100]` on a 5-element array doesn't raise an error, it reads whatever garbage byte
happens to sit at that memory address (or crashes the whole program — "segmentation fault").

**Python mindset:** Indexing is bounds-checked and raises a clear `IndexError` instead of
corrupting memory or crashing. Python also supports **negative indices** and **slicing**
(`start:stop:step`), which C has no native syntax for at all.

In [ ]:
nums = [10, 20, 30, 40, 50]

print(nums[0])       # first
print(nums[-1])      # last  (no C equivalent without arr[len-1])
print(nums[1:3])     # slice: index 1 up to (not including) 3
print(nums[:3])      # from start
print(nums[2:])      # to end
print(nums[::2])     # every 2nd element
print(nums[::-1])    # reversed! very Pythonic trick

try:
    print(nums[10])
except IndexError as e:
    print("Error:", e, "<- Python protects you. C would NOT.")

In [ ]:
# Slicing also works on strings and tuples (any sequence)
s = "Hello, World!"
print(s[7:12])
print(s[::-1])       # reverse a string in one line

# Slice assignment - only lists support this (they're mutable)
nums = [1, 2, 3, 4, 5]
nums[1:3] = [20, 30, 40]   # replace a slice with a different-length slice!
print(nums)

**Checkpoint:** What happens in C if you read past the end of an array? What happens in Python?

## 8. Nested Data Structures

**C mindset:** Nesting requires you to think in terms of pointers-to-pointers
(`int** matrix`) and manually allocate each row.

**Python mindset:** Since every Python variable is really a *reference* to an object, you can
nest lists inside lists, dicts inside lists, lists inside dicts — freely, with no extra ceremony.
But because they're references, **copying nested structures needs care** (shallow vs deep copy).

In [ ]:
# A matrix as a list of lists
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]
print(matrix[1][2])   # row 1, col 2 -> 6

# A more realistic nested structure: list of dicts
students = [
    {"name": "Asha", "marks": [90, 85, 88]},
    {"name": "Ravi", "marks": [70, 75, 80]},
]
for s in students:
    avg = sum(s["marks"]) / len(s["marks"])
    print(f"{s['name']}: avg = {avg:.1f}")

In [1]:
import copy

original = {"name": "Deepak", "skills": ["Python", "FastAPI"]}

shallow = original.copy()          # copies the outer dict, but inner list is SHARED
shallow["skills"].append("Docker")
print("original after shallow copy edit:", original)   # changed too! shared inner list

deep = copy.deepcopy(original)     # copies EVERYTHING, recursively, no sharing
deep["skills"].append("AWS")
print("original after deep copy edit:", original)       # untouched this time
print("deep:", deep)

original after shallow copy edit: {'name': 'Deepak', 'skills': ['Python', 'FastAPI', 'Docker']}
original after deep copy edit: {'name': 'Deepak', 'skills': ['Python', 'FastAPI', 'Docker']}
deep: {'name': 'Deepak', 'skills': ['Python', 'FastAPI', 'Docker', 'AWS']}


**Checkpoint:** What's the difference between a shallow copy and a deep copy? When would a shallow copy bite you?

## 9. Functions

**C mindset:** Functions must be declared with an exact return type and parameter types before
use (a "function prototype"). Functions are not values — you can't easily pass a function around
without special "function pointer" syntax.

**Python mindset:** Functions are **first-class objects** — no type declarations needed, and you
can assign them to variables, pass them as arguments, return them from other functions, and store
them in lists/dicts, just like any other value.

In [ ]:
def greet(name):
    return f"Hello, {name}!"

print(greet("Deepak"))

# Functions are objects: you can assign them, pass them around
say_hi = greet          # no parentheses -> we're referring to the function itself
print(say_hi("Chand"))

def apply_twice(func, value):
    return func(func(value))

def add_one(x):
    return x + 1

print(apply_twice(add_one, 5))   # 5 -> 6 -> 7

**Checkpoint:** What does it mean to say Python functions are 'first-class objects'? Give one thing you can do with a function in Python that's awkward in C.

## 10. Parameters

**C mindset:** C is strictly **pass-by-value**. If you want a function to modify a variable in
the caller, you must manually pass a *pointer* (`&x`) to it.
```c
void increment(int *x) { *x = *x + 1; }   // need a pointer to actually change the caller's variable
```

**Python mindset:** Python passes **object references by value** — sometimes called
"pass by object reference" or "call by sharing". The function gets a reference to the *same*
object. Whether that lets you "mutate the caller's data" depends entirely on whether the object
itself is mutable (see Section 6!).

In [ ]:
# Immutable argument: function CANNOT change caller's variable
def try_increment(x):
    x = x + 1        # rebinds the local name x to a NEW int; doesn't touch caller's variable
    return x

n = 5
try_increment(n)
print("n after try_increment:", n)     # unchanged: 5

# Mutable argument: function CAN change caller's object, since both point to same list
def add_item(lst):
    lst.append("new item")   # mutates the SAME list object the caller has

my_list = ["a", "b"]
add_item(my_list)
print("my_list after add_item:", my_list)   # changed! ['a', 'b', 'new item']

In [ ]:
# Default values, keyword arguments, positional arguments
def describe_pet(name, species="dog", age=1):
    print(f"{name} is a {age}-year-old {species}")

describe_pet("Rex")                         # use defaults for species/age
describe_pet("Whiskers", species="cat")     # keyword argument
describe_pet(age=3, name="Tom", species="cat")   # order doesn't matter with keywords

# Positional-only vs keyword-only enforcement (Python 3.8+)
def strict_example(a, b, /, c, *, d):
    # a, b: must be positional
    # c: either
    # d: must be keyword
    return a + b + c + d

print(strict_example(1, 2, 3, d=4))

**Checkpoint:** If you pass a list into a function and the function does `.append()` on it, does the caller's list change? What if the function does `list_param = []` instead?

## 11. Return Values

**C mindset:** A function can return exactly **one value** of one declared type. To "return"
multiple values, you need pointers (out-parameters) or a `struct`.

**Python mindset:** A function can return **multiple values at once** — Python just packs them
into a tuple automatically, and you unpack them on the caller's side.

In [ ]:
def min_max(numbers):
    return min(numbers), max(numbers)   # actually returns a TUPLE (min, max)

lo, hi = min_max([4, 1, 9, 2, 7])   # tuple unpacking
print(f"min={lo}, max={hi}")

result = min_max([4, 1, 9, 2, 7])
print(result, type(result))

def maybe_find(lst, target):
    if target in lst:
        return lst.index(target)
    return None            # functions with no explicit return (or bare `return`) give None

print(maybe_find([1, 2, 3], 2))
print(maybe_find([1, 2, 3], 99))

**Checkpoint:** How does Python let a function 'return two values' when C functions can only return one?

## 12. Scope

**C mindset:** Scope is mostly block-based (`{ }`) and file-based (`static` for file-private
globals, `extern` to share across files). Fairly mechanical.

**Python mindset:** Python follows the **LEGB rule** when looking up a name:
**L**ocal → **E**nclosing (an outer function) → **G**lobal (module level) → **B**uilt-in
(`len`, `print`, etc). Python does NOT create a new scope for `if`/`for`/`while` blocks like C
does for `{ }` — only functions (and classes, lambdas) create new scopes.

In [ ]:
x = "global x"

def outer():
    x = "enclosing x"
    def inner():
        x = "local x"
        print("inner sees:", x)
    inner()
    print("outer sees:", x)

outer()
print("module level sees:", x)

In [ ]:
counter = 0

def increment_broken():
    counter = counter + 1   # ERROR: Python sees an assignment to counter in this
                             # function, so it treats counter as LOCAL for the
                             # ENTIRE function body -> reading it before assignment fails
try:
    increment_broken()
except UnboundLocalError as e:
    print("Error:", e)

def increment_fixed():
    global counter           # explicitly say "I mean the global one"
    counter = counter + 1

increment_fixed()
print("counter:", counter)

# `nonlocal` is the equivalent for enclosing (not global) scope
def outer():
    count = 0
    def inner():
        nonlocal count
        count += 1
    inner()
    inner()
    print("count:", count)

outer()

In [ ]:
# Unlike C, if/for/while do NOT create a new scope in Python
if True:
    y = "defined inside an if block"
print(y)   # still accessible! no block scoping for if/for/while

for i in range(3):
    pass
print("i survives the loop:", i)   # 2, the loop var leaks into the enclosing scope

**Checkpoint:** What's the LEGB rule? Why does `increment_broken()` raise an error while `increment_fixed()` doesn't?

## 13. `*args` and `**kwargs`

**C mindset:** C does have variadic functions (`printf(const char *fmt, ...)`) using
`<stdarg.h>`, but it's unsafe and manual — the function has no idea how many arguments were
actually passed or their types; it relies on YOU telling it (e.g. via the format string) and
walks raw memory with `va_arg`.

**Python mindset:** `*args` collects any number of extra **positional** arguments into a tuple.
`**kwargs` collects any number of extra **keyword** arguments into a dict. Totally safe, typed,
introspectable.

In [ ]:
def add_all(*args):          # args is a TUPLE of whatever positional args were passed
    print("args:", args, type(args))
    return sum(args)

print(add_all(1, 2, 3))
print(add_all(1, 2, 3, 4, 5))
print(add_all())

def describe(**kwargs):      # kwargs is a DICT of whatever keyword args were passed
    print("kwargs:", kwargs, type(kwargs))
    for key, value in kwargs.items():
        print(f"  {key} = {value}")

describe(name="Deepak", role="developer", city="Hyderabad")

In [ ]:
# Combining everything: standard order is
# def f(positional, *args, keyword_only=default, **kwargs):
def full_example(a, b, *args, mode="default", **kwargs):
    print("a:", a, "b:", b)
    print("extra positional (args):", args)
    print("mode:", mode)
    print("extra keyword (kwargs):", kwargs)

full_example(1, 2, 3, 4, mode="custom", extra="yes", debug=True)

# The famous "unpacking" use of * and ** when CALLING a function
def add3(x, y, z):
    return x + y + z

nums = [1, 2, 3]
print(add3(*nums))          # unpacks list -> add3(1, 2, 3)

kwargs_dict = {"x": 10, "y": 20, "z": 30}
print(add3(**kwargs_dict))  # unpacks dict -> add3(x=10, y=20, z=30)

**Checkpoint:** What Python type does `*args` collect into? What about `**kwargs`? Why is this safer than C's variadic `printf`-style functions?

## 14. Comprehensions

**C mindset:** No such concept exists. Building a filtered/transformed array always means an
explicit loop, a manually managed output array/index counter.

**Python mindset:** Comprehensions let you build a list/set/dict in **one readable expression**
instead of a multi-line loop. Under the hood Python is still looping — comprehensions are
"syntactic sugar", not magic — but they express *intent* much more directly.

In [ ]:
# The "C way" (explicit loop) vs the "Pythonic way" (comprehension) - same result

# Squares of even numbers from 0-9
squares_loop = []
for n in range(10):
    if n % 2 == 0:
        squares_loop.append(n ** 2)
print("loop version:  ", squares_loop)

squares_comp = [n ** 2 for n in range(10) if n % 2 == 0]
print("comprehension: ", squares_comp)

# Set comprehension
unique_lengths = {len(word) for word in ["hi", "hello", "hey", "yo"]}
print(unique_lengths)

# Dict comprehension
word_lengths = {word: len(word) for word in ["python", "is", "fun"]}
print(word_lengths)

# Nested comprehension - flatten a matrix
matrix = [[1, 2], [3, 4], [5, 6]]
flat = [num for row in matrix for num in row]
print(flat)

**Checkpoint:** Rewrite `[x*2 for x in range(5) if x != 2]` as an explicit for-loop. What does it produce?

## 15. `enumerate`

**C mindset:** Looping with an index is the *only* way to loop in C — you always manually manage
an integer counter (`for (int i = 0; i < n; i++)`).

**Python mindset:** Python's `for` loop is really a `for-each` (it iterates *values*, not
indices). `enumerate()` is what you reach for when you need **both** the index and the value.

In [ ]:
fruits = ["apple", "banana", "cherry"]

# Without enumerate (clunky, "C-style" index management)
for i in range(len(fruits)):
    print(i, fruits[i])

print("---")

# With enumerate (Pythonic)
for i, fruit in enumerate(fruits):
    print(i, fruit)

print("---")

# enumerate can start counting from anywhere
for i, fruit in enumerate(fruits, start=1):
    print(f"{i}. {fruit}")

**Checkpoint:** What two things does `enumerate()` give you on each iteration?

## 16. `zip`

**C mindset:** To walk two arrays together, you use one shared index and manually bounds-check
against BOTH array lengths (`for (i=0; i < min(len1,len2); i++)`).

**Python mindset:** `zip()` pairs up multiple iterables element-by-element, stopping at the
**shortest** one automatically.

In [ ]:
names = ["Deepak", "Asha", "Ravi"]
scores = [90, 85, 78]

for name, score in zip(names, scores):
    print(f"{name}: {score}")

# zip stops at the shortest iterable
extra_names = ["Deepak", "Asha", "Ravi", "Extra Person"]
print(list(zip(extra_names, scores)))   # "Extra Person" is silently dropped

# zip can combine more than 2 iterables
cities = ["Hyderabad", "Delhi", "Mumbai"]
combined = list(zip(names, scores, cities))
print(combined)

# unzip using zip(*...)
pairs = [("a", 1), ("b", 2), ("c", 3)]
letters, numbers = zip(*pairs)
print(letters, numbers)

# Building a dict from two parallel lists - very common zip use case
name_to_score = dict(zip(names, scores))
print(name_to_score)

**Checkpoint:** If one list has 5 items and another has 3, how many pairs does `zip()` produce?

## 17. `map`

**C mindset:** Applying a transformation to every element of an array means writing a loop that
walks the array and writes results into a (separately allocated) output array.

**Python mindset:** `map(function, iterable)` applies `function` to every element and returns a
**lazy iterator** (values are computed one at a time, only when needed) — wrap it in `list()` to
see all results immediately.

In [ ]:
nums = [1, 2, 3, 4, 5]

def square(x):
    return x ** 2

squared = map(square, nums)
print(squared, type(squared))     # a map object - LAZY, not computed yet!
print(list(squared))              # forces evaluation

# map with a lambda (anonymous function) - very common pairing
doubled = list(map(lambda x: x * 2, nums))
print(doubled)

# map over multiple iterables at once
a = [1, 2, 3]
b = [10, 20, 30]
sums = list(map(lambda x, y: x + y, a, b))
print(sums)

# Equivalent list comprehension (often preferred for readability in modern Python)
print([x * 2 for x in nums])

**Checkpoint:** Why does `print(map(square, nums))` NOT show `[1, 4, 9, 16, 25]` directly? What makes `map` 'lazy'?

## 18. `filter`

**C mindset:** Keeping only elements that satisfy a condition means a loop, an `if`, and manually
copying qualifying elements into a new (separately sized/allocated) array.

**Python mindset:** `filter(function, iterable)` keeps only the elements where `function(element)`
is truthy — also a lazy iterator, just like `map`.

In [ ]:
nums = list(range(1, 11))

def is_even(n):
    return n % 2 == 0

evens = filter(is_even, nums)
print(list(evens))

# filter with lambda
odds = list(filter(lambda n: n % 2 != 0, nums))
print(odds)

# filter(None, iterable) drops "falsy" values (0, "", None, False, [])
messy = [0, 1, "", "hello", None, True, [], [1, 2]]
print(list(filter(None, messy)))

# Equivalent list comprehension
print([n for n in nums if n % 2 == 0])

# Chaining map + filter: square only the even numbers
result = list(map(lambda n: n ** 2, filter(is_even, nums)))
print(result)

**Checkpoint:** What does `filter(None, some_list)` do?

---
## 📝 Week Assessment

### Part A — Oral / Conceptual Questions
*(Try to answer out loud or in a sentence or two, without running code. This checks understanding, not memorization.)*

1. In one sentence, what is the core difference between how C and Python manage memory?
2. Name two mutable and two immutable built-in Python types.
3. Why can a `tuple` be a dictionary key but a `list` cannot?
4. What does "pass by object reference" mean, and how is it different from C's "pass by value"?
5. What is the LEGB rule?
6. Why is the mutable-default-argument (`def f(x, lst=[])`) pattern dangerous?
7. What's the difference between a shallow copy and a deep copy?
8. What does `*args` collect into, and what does `**kwargs` collect into?
9. Why are `map()` and `filter()` described as "lazy"?
10. What does Python do automatically that C forces the programmer to do manually, when it comes to array/list bounds?

### Part B — Coding Exercises
Write your solution in the empty cell under each question. Solutions are in the hidden section
at the very bottom — try first before peeking!

**Exercise 1.** Write a function `word_frequencies(text)` that takes a sentence and returns a dictionary mapping each word (lowercased) to how many times it appears.

In [ ]:
# Your code here
def word_frequencies(text):
    pass

# test
print(word_frequencies("the cat sat on the mat the cat ran"))

**Exercise 2.** Using a list comprehension, given `nums = [3, -1, 4, -5, 9, -2]`, produce a list containing only the positive numbers, squared.

In [ ]:
# Your code here
nums = [3, -1, 4, -5, 9, -2]

result = None   # replace with your comprehension
print(result)   # expect [9, 16, 81]

**Exercise 3.** Write a function `make_multiplier(n)` that returns a *new function* which multiplies its argument by `n`. (Tests: functions as first-class objects + closures/scope.)

In [ ]:
# Your code here
def make_multiplier(n):
    pass

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(5))   # expect 10
print(triple(5))   # expect 15

**Exercise 4.** Given two lists `names` and `ages`, use `zip` and a dict comprehension to build `{name: age}`, then use `enumerate` to print each pair numbered starting at 1.

In [ ]:
names = ["Deepak", "Asha", "Ravi"]
ages = [24, 22, 27]

# Your code here
name_age = None   # dict comprehension using zip

# print numbered list using enumerate

**Exercise 5.** Write a function `safe_divide(a, b, *, default=None)` that returns `a / b`, but returns `default` instead of crashing if `b` is 0. (Tests: keyword-only args + default params.)

In [ ]:
# Your code here
def safe_divide(a, b, *, default=None):
    pass

print(safe_divide(10, 2))     # 5.0
print(safe_divide(10, 0))     # None
print(safe_divide(10, 0, default=-1))   # -1

**Exercise 6.** Demonstrate the mutability trap: write two small functions, `bad_append` (which reassigns a new local list instead of modifying the caller's list) and `good_append` (which actually mutates the caller's list with `.append`). Show, with print statements, that only `good_append`'s effect is visible to the caller.

In [ ]:
# Your code here

---
### Solutions (no peeking until you've tried!)

<details>
<summary>Click to expand solutions</summary>

```python
# Exercise 1
def word_frequencies(text):
    freq = {}
    for word in text.lower().split():
        freq[word] = freq.get(word, 0) + 1
    return freq

# Exercise 2
nums = [3, -1, 4, -5, 9, -2]
result = [n ** 2 for n in nums if n > 0]

# Exercise 3
def make_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

# Exercise 4
names = ["Deepak", "Asha", "Ravi"]
ages = [24, 22, 27]
name_age = {name: age for name, age in zip(names, ages)}
for i, (name, age) in enumerate(name_age.items(), start=1):
    print(f"{i}. {name} - {age}")

# Exercise 5
def safe_divide(a, b, *, default=None):
    if b == 0:
        return default
    return a / b

# Exercise 6
def bad_append(lst):
    lst = lst + ["new"]   # rebinds local name -> caller's list untouched

def good_append(lst):
    lst.append("new")     # mutates the SAME object the caller has

original = [1, 2, 3]
bad_append(original)
print("after bad_append:", original)   # unchanged

good_append(original)
print("after good_append:", original)  # changed
```

</details>